## 1. Imports

In [1]:
import sys
from pathlib import Path

# Add parent directory (where utility.py lives) to the path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from utility import _feature_engineering, setpoint_df, FeatureCreator
from config import CONTROL_VARS
from data_cleaning import (
    unique_in_order, ordered_difference, ordered_intersection,
)

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr  # Changed import path

sns.set(rc={"figure.figsize": (12, 4)})
sns.set_style("whitegrid")
sns.set_context("notebook")

## 2. Configuration

In [2]:
Y_COLUMN = "MBS_SCT_CD"

DATA_PATH = "../data/costimier_turnup.parquet"

CREATED_VARIABLE_CANDIDATES = [
    "delta_basis_weight",
    "Starch_uptake__g/m2_",
    "Water_flow_Predryer",
    "Water_flow_Afterdryer",
    "Water_flow",
    "flow_diluted_starch",
    "Fibre__g/m2_",
    "Water_flow_Afterdryer_input", 
    "Water_flow_Afterdryer_output",
    "dewatering",
    "fibre_short/long"
]

## 3. Data Loading & Filtering

In [3]:
# Load raw data
turnup_data = pd.read_parquet(DATA_PATH)

ctl_vars = unique_in_order(
    v for v in CONTROL_VARS[Y_COLUMN] if "vacuum" not in v.lower()
)
created_vars = ordered_intersection(CREATED_VARIABLE_CANDIDATES, ctl_vars)

turnup_data = _feature_engineering(turnup_data, setpoint_df, steam_null=False, clip=False)
turnup_data = turnup_data.set_index("Wedge_Time").sort_index()

if len(created_vars)>0:
    fc = FeatureCreator(features_to_create=created_vars, features_to_keep= list(set(turnup_data.columns.to_list()+created_vars)))
    turnup_data = fc.fit_transform(turnup_data)
    fr = fc.features_required()
else:
    fr=[]

var_names = [Y_COLUMN] +  unique_in_order(ordered_difference(ctl_vars, created_vars) + fr)
data_array = turnup_data[var_names].dropna().values
dataframe = pp.DataFrame(data_array, var_names=var_names)



In [4]:
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr
import tigramite.plotting as tp
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Conditional-independence test
# ------------------------------------------------------------
parcorr = ParCorr(significance="analytic")


# ------------------------------------------------------------
# 2. PCMCI object
# ------------------------------------------------------------
pcmci = PCMCI(
    dataframe=dataframe,
    cond_ind_test=parcorr,
    verbosity=1,
)


# ------------------------------------------------------------
# 3. Run PCMCI+
#
# tau_min = 0 allows contemporaneous relationships
# tau_max = 1 also allows previous-reel relationships
# ------------------------------------------------------------
results = pcmci.run_pcmciplus(
    tau_min=0,
    tau_max=1,
    pc_alpha=0.05,
)


# ------------------------------------------------------------
# 4. Inspect result matrices
# ------------------------------------------------------------
graph = results["graph"]
val_matrix = results["val_matrix"]
p_matrix = results["p_matrix"]

print("graph shape:", graph.shape)
print("val_matrix shape:", val_matrix.shape)
print("p_matrix shape:", p_matrix.shape)




##
## Step 1: PC1 algorithm for selecting lagged conditions
##

Parameters:
independence test = par_corr
tau_min = 1
tau_max = 1
pc_alpha = [0.05]
max_conds_dim = None
max_combinations = 1



## Resulting lagged parent (super)sets:

    Variable MBS_SCT_CD has 8 link(s):
        (MBS_SCT_CD -1): max_pval = 0.00000, |min_val| =  0.395
        (Current_basis_weight -1): max_pval = 0.00000, |min_val| =  0.060
        (Starch_uptake_by_paper_Top_Roll__g/m2_ -1): max_pval = 0.00000, |min_val| =  0.060
        (Starch_uptake_by_paper_Bottom_Roll__g/m2_ -1): max_pval = 0.00000, |min_val| =  0.055
        (Headbox_consistency -1): max_pval = 0.00000, |min_val| =  0.048
        (Short_fibre_flow -1): max_pval = 0.00001, |min_val| =  0.045
        (Bentonite_1_mass_flow__g/T_ -1): max_pval = 0.03926, |min_val| =  0.021
        (grammage -1): max_pval = 0.04207, |min_val| =  0.021

    Variable retention has 9 link(s):
        (retention -1): max_pval = 0.00000, |min_val| =  0.779
        (Speed

MemoryError: Unable to allocate 1.91 MiB for an array with shape (9616, 26) and data type float64

In [1]:
tp.plot_graph(
    graph=results["graph"],
    val_matrix=results["val_matrix"],
    var_names=var_names,
    link_colorbar_label="Partial correlation",
    node_colorbar_label="Autodependency",
)

plt.show()

NameError: name 'tp' is not defined